# GLF — BM25, E5 multilingüe y recuperación híbrida en Colab

Notebook autónomo: contiene el código necesario y no depende de que GitHub esté autorizado.

**Estado:** preparado para ejecución; no incluye salidas inventadas. E5 es un modelo preentrenado, no un experimento de entrenamiento. Este notebook no sustituye `overfitting_analysis.ipynb` de semana 3.

En Colab: Archivo → Subir notebook. Ejecutar las celdas en orden. CPU funciona; GPU es opcional y su disponibilidad no está garantizada. Usar solo datos autorizados para procesarse en Google Colab. El ZIP se carga en el almacenamiento temporal de la sesión, sin montar Drive.

Modelo: https://huggingface.co/intfloat/multilingual-e5-small


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers>=3,<6', 'huggingface-hub>=0.24,<2', 'numpy>=1.26,<3'])


In [ ]:
import os, json, hashlib, platform, importlib.metadata, time
from pathlib import Path
import numpy as np
import torch
np.random.seed(42)
torch.manual_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RUN = Path.cwd() / 'glf_colab_run'
RUN.mkdir(exist_ok=True)
versions = {p: importlib.metadata.version(p) for p in ['sentence-transformers','transformers','torch','numpy','huggingface-hub']}
print('Dispositivo:', DEVICE, '\nVersiones:', versions)


## 1. Código del proyecto incluido
Se guarda dentro de `glf_colab_run`. Permite reproducir esta versión sin descargar código del repositorio.

In [ ]:
SOURCES = {'src/__init__.py': '"""Herramientas reproducibles para el proyecto GLF."""\n', 'src/eda.py': '"""EDA de agregados sin exportar texto ni identificadores personales."""\nimport argparse\nimport collections\nimport json\nfrom pathlib import Path\nimport zipfile\n\n\ndef load_records(archive, member):\n    """Leer JSONL UTF-8 sin separar los caracteres Unicode internos del texto."""\n    with zipfile.ZipFile(archive) as source:\n        return [json.loads(line) for line in source.read(member).decode(\'utf-8-sig\').split(\'\\n\') if line.strip()]\n\n\ndef summarize(archive):\n    """Calcular estadísticas agregadas de documentos, pares y fragmentos."""\n    docs = load_records(archive, \'corpus_documentos.jsonl\')\n    chunks = load_records(archive, \'corpus_chunks_rag.jsonl\')\n    pairs = load_records(archive, \'expedientes_pareados.jsonl\')\n    if not docs or not chunks or not pairs:\n        raise ValueError(\'El corpus requiere documentos, fragmentos y expedientes.\')\n    ids = {d[\'document_id\'] for d in docs}\n    if len(ids) != len(docs) or any(c[\'document_id\'] not in ids for c in chunks):\n        raise ValueError(\'Identificadores documentales duplicados o referencias rotas.\')\n    splits = collections.defaultdict(set)\n    for row in docs + chunks:\n        splits[row[\'project_code\']].add(row[\'split\'])\n    if any(len(values) != 1 for values in splits.values()):\n        raise ValueError(\'Un expediente aparece en particiones diferentes.\')\n    count = lambda rows, key: dict(sorted(collections.Counter(r[key] for r in rows).items()))\n    lengths = [len(c[\'text\'].split()) for c in chunks]\n    bins = {\'1-249\': 0, \'250-499\': 0, \'500-749\': 0, \'750 o más\': 0}\n    for n in lengths:\n        if n <= 0:\n            raise ValueError(\'Existe un fragmento vacío.\')\n        bins[\'1-249\' if n < 250 else \'250-499\' if n < 500 else \'500-749\' if n < 750 else \'750 o más\'] += 1\n    return {\'documents\': len(docs), \'projects\': len(pairs), \'chunks\': len(chunks),\n            \'document_words\': sum(len(d[\'text\'].split()) for d in docs),\n            \'roles\': count(docs, \'document_role\'), \'languages\': count(docs, \'language\'),\n            \'groups\': count(docs, \'source_group\'), \'project_splits\': count(pairs, \'split\'),\n            \'chunk_splits\': count(chunks, \'split\'), \'chunk_lengths\': bins}\n\n\ndef figures(metrics, output):\n    """Crear seis figuras vectoriales con ReportLab, usando solo agregados."""\n    from reportlab.graphics import renderSVG\n    from reportlab.graphics.charts.barcharts import HorizontalBarChart\n    from reportlab.graphics.shapes import Drawing, String\n    from reportlab.lib import colors\n    specs = [(\'roles\', \'Documentos por función\'), (\'languages\', \'Documentos por idioma declarado\'),\n             (\'groups\', \'Documentos por grupo\'), (\'project_splits\', \'Expedientes por partición\'),\n             (\'chunk_splits\', \'Fragmentos por partición\'), (\'chunk_lengths\', \'Longitud de fragmentos en palabras\')]\n    output.mkdir(parents=True, exist_ok=True)\n    for i, (key, title) in enumerate(specs, 1):\n        values = metrics[key]\n        d = Drawing(640, 260)\n        d.add(String(20, 237, title, fontName=\'Helvetica-Bold\', fontSize=14))\n        chart = HorizontalBarChart()\n        chart.x, chart.y, chart.width, chart.height = 200, 40, 370, 170\n        chart.data = [list(values.values())]\n        chart.categoryAxis.categoryNames = list(values)\n        chart.categoryAxis.labels.fontSize = 10\n        chart.valueAxis.valueMin = 0\n        chart.valueAxis.valueMax = max(values.values()) * 1.2\n        chart.bars[0].fillColor = colors.HexColor(\'#255f70\')\n        chart.barLabelFormat = \'%d\'\n        chart.barLabels.boxAnchor = \'w\'\n        chart.barLabels.dx = 4\n        d.add(chart)\n        d.add(String(200, 13, \'Fuente: dataset GLF. Recuentos calculados; no son resultados del RAG.\', fontSize=8))\n        renderSVG.drawToFile(d, str(output / f\'{i:02d}_{key}.svg\'))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--archive\', required=True, type=Path)\n    parser.add_argument(\'--output\', default=Path(\'results\'), type=Path)\n    args = parser.parse_args()\n    metrics = summarize(args.archive)\n    (args.output / \'metrics\').mkdir(parents=True, exist_ok=True)\n    (args.output / \'metrics\' / \'eda.json\').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    figures(metrics, args.output / \'figures\')\n    print(f"EDA completado: {metrics[\'documents\']} documentos, {metrics[\'projects\']} expedientes.")\n\n\nif __name__ == \'__main__\':\n    main()\n', 'src/retrieval.py': '"""Recuperador lexical BM25. No genera respuestas ni estima confianza."""\nimport math\nimport re\nimport unicodedata\nfrom collections import Counter\nfrom .eda import load_records\n\nSTOP = set(\'a al de del el la las los un una unos unas y o en por para con que se es sobre como cuales\'.split())\n\n\ndef tokens(text):\n    value = \'\'.join(c for c in unicodedata.normalize(\'NFKD\', text.lower()) if not unicodedata.combining(c))\n    return [t for t in re.findall(r\'[a-z0-9]+\', value) if t not in STOP and len(t) > 1]\n\n\nclass BM25:\n    def __init__(self, records):\n        self.records = records\n        if not records or any(not r.get(\'text\', \'\').strip() for r in records):\n            raise ValueError(\'Corpus vacío o con fragmentos sin texto.\')\n        self.counts = [Counter(tokens(r[\'text\'])) for r in records]\n        self.lengths = [sum(c.values()) for c in self.counts]\n        self.average = sum(self.lengths) / len(records)\n        if not self.average:\n            raise ValueError(\'Corpus sin términos indexables.\')\n        df = Counter(t for c in self.counts for t in c)\n        self.idf = {t: math.log(1 + (len(records) - n + .5) / (n + .5)) for t, n in df.items()}\n\n    def search(self, query, limit=5):\n        terms = set(tokens(query))\n        scores = []\n        for i, counts in enumerate(self.counts):\n            score = sum(self.idf.get(t, 0) * counts[t] * 2.5 /\n                        (counts[t] + 1.5 * (.25 + .75 * self.lengths[i] / self.average))\n                        for t in terms)\n            if score > 0:\n                scores.append((score, i))\n        return [dict(self.records[i], score=round(s, 5)) for s, i in sorted(scores, key=lambda x: (-x[0], x[1]))[:limit]]\n\n\ndef load_spanish(archive):\n    member = \'work/GLF_SGAS_Corpus_ES/04_corpus_rag/Corpus_GLF_SGAS_ES.jsonl\'\n    records = load_records(archive, member)\n    if any(r.get(\'language\') != \'es\' for r in records):\n        raise ValueError(\'El corpus seleccionado debe estar en español.\')\n    return BM25(records)\n', 'src/semantic.py': '"""E5 opcional y fusión RRF; no sustituye validación de relevancia."""\nimport numpy as np\n\n\ndef passages(text, tokenizer, max_tokens=512):\n    """Dividir por palabras hasta que cada pasaje quepa; evitar truncado silencioso."""\n    words = text.split()\n    if not words:\n        raise ValueError(\'Pasaje vacío.\')\n    result = []\n    def divide(part):\n        value = \'passage: \' + \' \'.join(part)\n        if len(tokenizer.encode(value, add_special_tokens=True)) <= max_tokens:\n            result.append(value)\n        elif len(part) > 1:\n            middle = len(part) // 2\n            divide(part[:middle])\n            divide(part[middle:])\n        else:\n            raise ValueError(\'Una palabra excede el límite del modelo; revisar extracción.\')\n    divide(words)\n    return result\n\n\nclass E5:\n    name = \'E5-small\'\n    def __init__(self, records, model=None):\n        if not records:\n            raise ValueError(\'Corpus vacío.\')\n        if model is None:\n            try:\n                from sentence_transformers import SentenceTransformer\n            except ImportError as exc:\n                raise RuntimeError(\'Instalar las dependencias semánticas antes de usar E5.\') from exc\n            model = SentenceTransformer(\'intfloat/multilingual-e5-small\', device=\'cpu\', trust_remote_code=False)\n        self.model, self.records = model, records\n        self.model.max_seq_length = 512\n        self.owners, texts = [], []\n        for i, record in enumerate(records):\n            windows = passages(record[\'text\'], model.tokenizer)\n            texts.extend(windows)\n            self.owners.extend([i] * len(windows))\n        self.vectors = np.asarray(model.encode(texts, normalize_embeddings=True,\n                                               batch_size=16, show_progress_bar=False))\n\n    def search(self, query, limit=5):\n        value = \'query: \' + query.strip()\n        if not query.strip():\n            return []\n        if len(self.model.tokenizer.encode(value, add_special_tokens=True)) > 512:\n            raise ValueError(\'La consulta excede 512 tokens; acórtala.\')\n        vector = np.asarray(self.model.encode([value], normalize_embeddings=True, show_progress_bar=False))[0]\n        scores = self.vectors @ vector\n        best = {}\n        for owner, score in zip(self.owners, scores):\n            best[owner] = max(best.get(owner, float(\'-inf\')), float(score))\n        ranked = sorted(best, key=lambda i: (-best[i], i))[:limit]\n        return [dict(self.records[i], score=round(best[i], 6)) for i in ranked]\n\n\nclass Hybrid:\n    name = \'BM25+E5-RRF\'\n    def __init__(self, lexical, semantic, depth=20, constant=60):\n        if depth < 5 or constant <= 0:\n            raise ValueError(\'Parámetros RRF inválidos.\')\n        if [r[\'chunk_id\'] for r in lexical.records] != [r[\'chunk_id\'] for r in semantic.records]:\n            raise ValueError(\'Los recuperadores deben compartir corpus y orden.\')\n        self.records = lexical.records\n        self.lexical, self.semantic = lexical, semantic\n        self.depth, self.constant = depth, constant\n\n    def search(self, query, limit=5):\n        scores, rows = {}, {}\n        for retriever in (self.lexical, self.semantic):\n            seen = set()\n            for rank, row in enumerate(retriever.search(query, max(limit, self.depth)), 1):\n                key = row[\'chunk_id\']\n                if key in seen:\n                    continue\n                seen.add(key)\n                rows[key] = row\n                scores[key] = scores.get(key, 0) + 1 / (self.constant + rank)\n        return [dict(rows[key], score=round(scores[key], 6))\n                for key in sorted(scores, key=lambda key: (-scores[key], key))[:limit]]\n', 'src/evaluation.py': '"""Evaluación macro de recuperación sobre etiquetas revisadas."""\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nfrom time import perf_counter\nfrom .retrieval import load_spanish\n\n\ndef metrics(retrieved, relevant, k=5):\n    if k < 1 or not relevant:\n        raise ValueError(\'Se requiere k positivo y al menos un fragmento relevante.\')\n    ranked = list(dict.fromkeys(retrieved))[:k]\n    gold = set(relevant)\n    hits = len(set(ranked) & gold)\n    return {\'recall_at_5\': hits / len(gold), \'precision_at_5\': hits / k,\n            \'hit_rate_at_5\': float(hits > 0)}\n\n\ndef evaluate(index, queries):\n    if not queries:\n        raise ValueError(\'No hay consultas para evaluar.\')\n    ids = set()\n    known = {r[\'chunk_id\'] for r in index.records}\n    # Validar todas las etiquetas antes de ejecutar consultas.\n    for q in queries:\n        if not isinstance(q, dict) or not isinstance(q.get(\'query_id\'), str) or not q[\'query_id\'].strip():\n            raise ValueError(\'Falta query_id.\')\n        if q[\'query_id\'] in ids:\n            raise ValueError(\'Identificador de consulta duplicado.\')\n        ids.add(q[\'query_id\'])\n        if q.get(\'review_status\') != \'approved\' or not str(q.get(\'reviewer\', \'\')).strip():\n            raise ValueError(\'Todas las consultas requieren revisión aprobada y responsable.\')\n        if not isinstance(q.get(\'query\'), str) or not q[\'query\'].strip():\n            raise ValueError(\'Consulta vacía.\')\n        gold = q.get(\'relevant_chunk_ids\')\n        if not isinstance(gold, list) or not gold or any(not isinstance(x, str) for x in gold):\n            raise ValueError(\'Se requieren identificadores relevantes revisados.\')\n        if len(set(gold)) != len(gold) or not set(gold) <= known:\n            raise ValueError(\'Etiquetas duplicadas o fragmentos ajenos al corpus.\')\n        if q.get(\'split\') not in (\'development\', \'validation\', \'test\'):\n            raise ValueError(\'Partición no válida.\')\n    splits = {q[\'split\'] for q in queries}\n    if len(splits) != 1:\n        raise ValueError(\'Evaluar una partición por ejecución.\')\n    rows = []\n    for q in queries:\n        started = perf_counter()\n        retrieved = [r[\'chunk_id\'] for r in index.search(q[\'query\'], limit=5)]\n        seconds = perf_counter() - started\n        rows.append(dict(query_id=q[\'query_id\'], retrieved_chunk_ids=retrieved,\n                         seconds=seconds, **metrics(retrieved, q[\'relevant_chunk_ids\'])))\n    times = sorted(r[\'seconds\'] for r in rows)\n    return {\'method\': getattr(index, \'name\', \'BM25\'), \'unit\': \'chunk\', \'aggregation\': \'macro\',\n            \'split\': next(iter(splits)), \'queries\': len(rows),\n            \'metrics\': {key: sum(r[key] for r in rows) / len(rows)\n                        for key in (\'recall_at_5\', \'precision_at_5\', \'hit_rate_at_5\')},\n            \'retrieval_only_p95_seconds\': times[math.ceil(.95 * len(times)) - 1],\n            \'latency_scope\': \'Solo búsqueda local; excluye carga, red, interfaz y generación. No verifica la meta p95 del sistema.\',\n            \'per_query\': rows}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--archive\', required=True, type=Path)\n    parser.add_argument(\'--labels\', required=True, type=Path)\n    parser.add_argument(\'--output\', required=True, type=Path)\n    parser.add_argument(\'--method\', choices=[\'bm25\', \'e5\', \'hybrid\'], default=\'bm25\')\n    args = parser.parse_args()\n    queries = [json.loads(line) for line in args.labels.read_text(encoding=\'utf-8-sig\').split(\'\\n\') if line.strip()]\n    index = load_spanish(args.archive)\n    if args.method != \'bm25\':\n        from .semantic import E5, Hybrid\n        semantic = E5(index.records)\n        index = semantic if args.method == \'e5\' else Hybrid(index, semantic)\n    report = evaluate(index, queries)\n    report[\'archive_sha256\'] = hashlib.sha256(args.archive.read_bytes()).hexdigest()\n    report[\'labels_sha256\'] = hashlib.sha256(args.labels.read_bytes()).hexdigest()\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Evaluación guardada. La latencia corresponde solo al recuperador.\')\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name, content in SOURCES.items():
    target = RUN / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
sys.path.insert(0, str(RUN))
from src.retrieval import load_spanish
from src.semantic import E5, Hybrid
from src.evaluation import evaluate


## 2. Cargar corpus
Seleccionar únicamente `GLF_SGAS_Corpus_ES.zip`. No cargar los expedientes privados para esta prueba normativa.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Selecciona exactamente un ZIP normativo.')
filename, data = next(iter(uploaded.items()))
if not filename.lower().endswith('.zip'):
    raise ValueError('Se requiere ZIP.')
archive = RUN / 'corpus.zip'
archive.write_bytes(data)
archive_sha = hashlib.sha256(data).hexdigest()
del uploaded, data
lexical = load_spanish(archive)
print('Fragmentos:', len(lexical.records), 'SHA256:', archive_sha)
assert len({r['chunk_id'] for r in lexical.records}) == len(lexical.records)


## 3. Descargar E5 e indexar
La primera ejecución descarga los pesos y puede tardar. Se registra la revisión exacta del modelo. Los textos largos se dividen sin descartar palabras y se conserva el identificador original. Se toma la mayor similitud de sus pasajes; esta decisión se debe evaluar.

In [ ]:
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer
MODEL_ID = 'intfloat/multilingual-e5-small'
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
started = time.perf_counter()
model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device=DEVICE, trust_remote_code=False)
semantic = E5(lexical.records, model=model)
hybrid = Hybrid(lexical, semantic)
index_seconds = time.perf_counter() - started
print('Revisión:', MODEL_REVISION)
print('Pasajes internos:', len(semantic.owners), 'Dimensiones:', semantic.vectors.shape)
print('Carga e indexación (s):', round(index_seconds, 2))


## 4. Búsqueda real con los tres métodos
Las puntuaciones de BM25, E5 y RRF tienen escalas diferentes. No son probabilidades ni confianza. Los resultados deben revisarse contra los documentos originales.

In [ ]:
QUERY = '¿Cómo debe atenderse una queja de las comunidades afectadas por un proyecto?'
retrievers = {'bm25': lexical, 'e5': semantic, 'hybrid': hybrid}
search_runs = {}
for method, index in retrievers.items():
    started = time.perf_counter()
    rows = index.search(QUERY, limit=5)
    elapsed = time.perf_counter() - started
    search_runs[method] = {'seconds': elapsed, 'chunk_ids': [r['chunk_id'] for r in rows]}
    print('\n', method.upper(), round(elapsed, 4), 's')
    for row in rows:
        print(row['chunk_id'], 'score=', row['score'])
        print(row['text'][:400], '\n')


## 5. Evaluación opcional con etiquetas humanas
Mantener `RUN_EVALUATION=False` hasta disponer de JSONL revisado: query_id, query, split, review_status="approved", reviewer, relevant_chunk_ids. No convertir los resultados del buscador en etiquetas automáticamente. Usar una sola partición; reservar prueba hasta finalizar ajustes.

In [ ]:
RUN_EVALUATION = False
reports = {}
if RUN_EVALUATION:
    reviewed = files.upload()
    if len(reviewed) != 1:
        raise ValueError('Selecciona un único JSONL de etiquetas revisadas.')
    label_bytes = next(iter(reviewed.values()))
    queries = [json.loads(line) for line in label_bytes.decode('utf-8-sig').split('\n') if line.strip()]
    for method, index in retrievers.items():
        reports[method] = evaluate(index, queries)
        reports[method]['labels_sha256'] = hashlib.sha256(label_bytes).hexdigest()
        print(method, reports[method]['metrics'])
else:
    print('Evaluación omitida: no se declaran Recall@5 ni cumplimiento de metas.')


## 6. Descargar registro de ejecución
Exporta versiones, revisión, identificadores recuperados y tiempos. No exporta textos ni pesos. Guarda también una copia del notebook con sus salidas después de revisarlas: las celdas de búsqueda muestran texto del corpus. La duración de una consulta no es p95 de la aplicación completa.

In [ ]:
record = {'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
          'versions': versions, 'python': platform.python_version(), 'device': DEVICE,
          'archive_sha256': archive_sha, 'chunks': len(lexical.records),
          'passages': len(semantic.owners), 'index_seconds': index_seconds,
          'search_runs': search_runs, 'evaluations': reports,
          'limitations': 'E5 preentrenado; no entrenamiento. Tiempos del recuperador, no de todo el sistema.'}
output = RUN / 'ejecucion_colab.json'
output.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
files.download(str(output))


## Conclusiones que debe completar el equipo
- ¿Qué evidencia relevante recupera cada método? Revisar el fragmento completo.
- ¿Qué errores aparecen y qué consultas no se resuelven?
- Si hay etiquetas: comparar resultados sobre la misma partición y corpus.
- No afirmar que se supera el 80 % ni que hay mejora sin medición validada.
- Este notebook cubre recuperación e inferencia; semana 3 aún requiere tarea entrenable, curvas y dos mejoras reales.
